# Orchestrator + UseModel Integration (Minimal Change)

This notebook combines only two existing workflows:

1. `Orchestrator_Pipeline.ipynb` for routing
2. `usemodelkaggle.ipynb` for model loading + inference

Target flow:
- User prompt
- Orchestrator routes to `script reviewer`.
- Impact Agent is not working in this orchestrator, but the router can tell it, and stop the running session in gradio
- UseModel pipeline runs and returns one final output

Running this Orchestrator:
- Open it in your VS Code.
- In the Extensions at the left bar, find `Colab` from `Google.com` (current install count is around 225,000), and install it.
- After installing, click the `Select Kernal` at the top right, then `Select Another Kernal`, and follow the sequence below:
  - `Colab`
  - `New Colab Server`
  - `GPU`
  - `A100` should be good
  - `High-RAM`
  - Give it a name, less than 10 characters. I used `A100-high`
  - Wait for a few seconds, then you can select the programming language (just like what you did in browser). Select `Python3 (ipykernel)`. 
  - START SURF!


In [2]:
# ============================================================
# STEP 0: Progress Checkpoint
# ============================================================
print("[STEP 0] Notebook loaded.")
print("[STEP 0.1] Next: install dependencies.")


[STEP 0] Notebook loaded.
[STEP 0.1] Next: install dependencies.


In [3]:
# ============================================================
# STEP 1: Install Dependencies (copy from existing notebooks)
# ============================================================
print("[STEP 1] Installing dependencies...")

!pip -q install -U google-genai gradio
!pip -q install -U google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip -q install -U unsloth unsloth-zoo modelscope peft accelerate bitsandbytes huggingface_hub python-docx PyPDF2

print("[STEP 1.1] Install complete.")


[STEP 1] Installing dependencies...
[STEP 1.1] Install complete.


In [4]:
# ============================================================
# STEP 2: Imports + Runtime Flags
# ============================================================
print("[STEP 2] Importing libraries...")

import os
import re
import torch
import getpass
import gradio as gr
from threading import Lock
from types import SimpleNamespace

from google import genai
from unsloth import FastLanguageModel
from peft import PeftModel

os.environ["UNSLOTH_DISABLE_LOG_STATS"] = "1"
os.environ.setdefault("UNSLOTH_USE_MODELSCOPE", "1")

print("[STEP 2.1] Imports ready.")


[STEP 2] Importing libraries...


/var/folders/d_/f3w5b1yn2_vd6lfw2ly0b8sc0000gn/T/ipykernel_17935/365652289.py:14: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

In [ ]:
# ============================================================
# STEP 3: Keys / Tokens
# ============================================================
print("[STEP 3] Loading credentials...")

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter GOOGLE_API_KEY: ")
    # <GOOGLE_API_KEY_REMOVED>
if not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN: ")
    # <HF_TOKEN_REMOVED>


print("[STEP 3.1] Credentials loaded.")


[STEP 3] Loading credentials...
[STEP 3.1] Credentials loaded.


In [ ]:
# ============================================================
# STEP 4: RequestOrchestrator (copied from Orchestrator_Pipeline)
# ============================================================
print("[STEP 4] Defining orchestrator class...")

class RequestOrchestrator:
    """
    Orchestrator that determines which agent should handle a user's request.
    Routes between Impact Agent and Script Reviewer.
    """

    IMPACT_AGENT = "impact agent"
    SCRIPT_REVIEWER = "script reviewer"

    def __init__(self, api_key: str, model_name: str = "gemini-2.5-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model_name = model_name
        print(f"[STEP 4.1] Orchestrator initialized with {model_name}")

    def _create_routing_prompt(self, user_prompt: str) -> str:
        system_prompt = f"""You are a routing system that determines which agent should handle a user's request.

You have TWO agents available:

1. **Impact Agent**: Handles questions about:
   - Impact, outcomes, effects, consequences, results
   - Benefits, changes, influence of actions/policies/programs
   - Analysis of societal, environmental, or business impacts
   - "What if" scenarios and their implications

2. **Script Reviewer**: Handles questions about:
   - Reviewing, analyzing, or critiquing scripts (movie, TV, theater)
   - Code review and programming help
   - Document review and editing
   - Writing feedback and improvements

USER QUERY: {user_prompt}

Based on the user's query, determine which agent should handle this request.

RESPOND WITH ONLY ONE OF THESE TWO OPTIONS (exactly as written):
- "impact agent"
- "script reviewer"

Your response should contain ONLY the agent name, nothing else."""
        return system_prompt

    def route_request(self, user_prompt: str) -> str:
        prompt = self._create_routing_prompt(user_prompt)
        try:
            response = self.client.models.generate_content(model=self.model_name, contents=prompt)
            decision = (getattr(response, "text", "") or "").strip().lower()
            if self.IMPACT_AGENT in decision:
                return self.IMPACT_AGENT
            elif self.SCRIPT_REVIEWER in decision:
                return self.SCRIPT_REVIEWER
            else:
                print(f"[STEP 4 WARN] Unexpected routing response: {decision}. Using fallback.")
                return self._fallback_routing(user_prompt)
        except Exception as e:
            print(f"[STEP 4 WARN] Routing error: {str(e)}. Using fallback.")
            return self._fallback_routing(user_prompt)

    def _fallback_routing(self, user_prompt: str) -> str:
        user_lower = user_prompt.lower()

        script_keywords = ["script", "review", "code", "analyze", "critique",
                          "feedback", "document", "written", "check", "screenplay",
                          "program", "function", "bug", "error", "syntax"]

        impact_keywords = ["impact", "effect", "outcome", "result", "consequence",
                          "benefit", "change", "influence", "affect", "what if",
                          "implications", "happens", "caused by"]

        script_score = sum(1 for keyword in script_keywords if keyword in user_lower)
        impact_score = sum(1 for keyword in impact_keywords if keyword in user_lower)

        if script_score > impact_score:
            return self.SCRIPT_REVIEWER
        else:
            return self.IMPACT_AGENT

print("[STEP 4.2] Orchestrator class ready.")


[STEP 4] Defining orchestrator class...
[STEP 4.2] Orchestrator class ready.


In [ ]:
# ============================================================
# STEP 5: Default User Prompt (for UI prefill)
# ============================================================
print("[STEP 5] Setting default prompt/context for Gradio UI...")

# This default text is prefilled in the input box.
# You can edit it in UI before clicking Analyze.
USER_PROMPT = "Please review my script opening and tell me what to improve."
ADDITIONAL_CONTEXT = "Focus on clarity, pacing, and emotional impact."

# Keep for backward compatibility with STEP 10 one-shot test.
# In Gradio flow, users upload files directly (no Drive file path needed here).
UPLOADED_FILE_PATH = None

print("[STEP 5.1] Default USER_PROMPT set.")
print("[STEP 5.2] Default CONTEXT set.")
print("[STEP 5.3] File will come from Gradio upload.")


[STEP 5] Setting default prompt/context for Gradio UI...
[STEP 5.1] Default USER_PROMPT set.
[STEP 5.2] Default CONTEXT set.
[STEP 5.3] File will come from Gradio upload.


In [ ]:
# ============================================================
# OPTIONAL: Gemini Connectivity Quick Check
# ============================================================
print("[OPTIONAL] Skip this cell if routing is already working.")
print("[OPTIONAL] This cell is intentionally lightweight to avoid long hangs.")


[OPTIONAL] Skip this cell if routing is already working.
[OPTIONAL] This cell is intentionally lightweight to avoid long hangs.


## Legacy Step (Skip)
This old Step 6 cell is deprecated.
Use the next code cell: **STEP 6: Route Prompt with Orchestrator (API timeout + fallback)**.


In [ ]:
# ============================================================
# STEP 6: Initialize Router (routing executes only on Analyze click)
# ============================================================
print("[STEP 6] Initializing orchestrator and safe_route...")

ROUTER_MODEL = os.getenv("ROUTER_MODEL", "gemini-2.5-flash-lite")
orchestrator = RequestOrchestrator(api_key=os.environ["GOOGLE_API_KEY"], model_name=ROUTER_MODEL)
print(f"[STEP 6.0] Router model: {ROUTER_MODEL}")


def keyword_route(prompt: str):
    text = (prompt or "").lower()

    script_keywords = [
        "script", "review", "screenplay", "dialogue", "plot", "character",
        "code", "bug", "function", "error", "syntax", "refactor", "document"
    ]
    impact_keywords = [
        "impact", "outcome", "consequence", "benefit", "harm", "society",
        "community", "ethics", "policy", "what if", "implication"
    ]

    script_score = sum(1 for k in script_keywords if k in text)
    impact_score = sum(1 for k in impact_keywords if k in text)

    if script_score > 0 and impact_score == 0:
        return orchestrator.SCRIPT_REVIEWER, script_score, impact_score
    if impact_score > 0 and script_score == 0:
        return orchestrator.IMPACT_AGENT, script_score, impact_score

    return None, script_score, impact_score


def safe_route(prompt: str, timeout_sec: int = 12):
    # 1) Fast keyword route first (always prompt-based)
    kw_agent, s_score, i_score = keyword_route(prompt)
    if kw_agent is not None:
        return kw_agent, f"keyword(script={s_score},impact={i_score})"

    # 2) LLM route only when keyword route is ambiguous
    try:
        routing_prompt = orchestrator._create_routing_prompt(prompt)
        response = orchestrator.client.models.generate_content(
            model=orchestrator.model_name,
            contents=routing_prompt,
        )
        decision = (getattr(response, "text", "") or "").strip().lower()

        if orchestrator.IMPACT_AGENT in decision:
            return orchestrator.IMPACT_AGENT, "llm"
        if orchestrator.SCRIPT_REVIEWER in decision:
            return orchestrator.SCRIPT_REVIEWER, "llm"

        fb = orchestrator._fallback_routing(prompt)
        return fb, f"fallback_format(script={s_score},impact={i_score})"

    except Exception as e:
        fb = orchestrator._fallback_routing(prompt)
        return fb, f"fallback_error:{type(e).__name__}(script={s_score},impact={i_score})"


print("[STEP 6.1] safe_route is ready.")
print("[STEP 6.2] Routing runs in STEP 11 when Analyze is clicked.")


[STEP 6] Initializing orchestrator and safe_route...
[STEP 4.1] Orchestrator initialized with gemini-2.5-flash-lite
[STEP 6.0] Router model: gemini-2.5-flash-lite
[STEP 6.1] safe_route is ready.
[STEP 6.2] Routing runs in STEP 11 when Analyze is clicked.


In [ ]:
# ============================================================
# STEP 7: Mount Google Drive + set adapter path
# ============================================================
print("[STEP 7] Mount Drive and configure adapter path...")

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[STEP 7.1] Drive mounted (or already mounted).")
except Exception as e:
    print("[STEP 7 WARN] Could not mount Drive in this runtime:", e)
finally:
    print("""
          ####################################################################################################################
          NOTE: You need to mount your Google Drive to access the the model files. 
          Before mounting, ensure that you have uploaded your model file to your Drive. 
          If you don't have the model files in Drive, it's located at https://drive.google.com/drive/folders/1uN9ZmE1PFcXRQQik_VAfoHFY1llRwO2I?usp=drive_link.
          ________________________________________________________________________________________________
          I put the model files in my Drive's 1st level (not in a subfolder) for easier access. 
          So after mounting, please modify the ADAPTER_DIR path below if your files are in a different location or subfolder.
          ####################################################################################################################
          """)

Mounted at /content/drive
[STEP 7.1] Drive mounted (or already mounted).

          ####################################################################################################################
          NOTE: You need to mount your Google Drive to access the the model files. 
          Before mounting, ensure that you have uploaded your model file to your Drive. 
          If you don't have the model files in Drive, it's located at https://drive.google.com/drive/folders/1uN9ZmE1PFcXRQQik_VAfoHFY1llRwO2I?usp=drive_link.
          ________________________________________________________________________________________________
          I put the model files in my Drive's 1st level (not in a subfolder) for easier access. 
          So after mounting, please modify the ADAPTER_DIR path below if your files are in a different location or subfolder.
          ####################################################################################################################
          

In [ ]:
ADAPTER_DIR = "/content/drive/MyDrive/my_finetuned_llm"   # Change this if your files are in a different location or subfolder in Drive.

print("[STEP 7.2] ADAPTER_DIR:", ADAPTER_DIR)
print("adapter exists =", os.path.exists(ADAPTER_DIR))
print("config exists =", os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")))

if not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    raise FileNotFoundError(
        f"adapter_config.json not found in {ADAPTER_DIR}. "
        "Please verify Drive mount and folder location."
    )

print("[STEP 7.3] Files in ADAPTER_DIR:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    print(f"  {f}")

[STEP 7.2] ADAPTER_DIR: /content/drive/MyDrive/my_finetuned_llm
adapter exists = True
config exists = True
[STEP 7.3] Files in ADAPTER_DIR:
  README.md
  adapter_config.json
  adapter_model.safetensors
  chat_template.jinja
  special_tokens_map.json
  tokenizer.json
  tokenizer_config.json


In [ ]:
# ============================================================
# STEP 8: Load Base Model + LoRA (copied from usemodelkaggle)
# ============================================================
print("[STEP 8] Loading fine-tuned model stack...")

BASE_ID = "unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit"
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Missing HF_TOKEN. Set it before running this cell.")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_ID,
        max_seq_length=2048,
        load_in_4bit=True,
        token=HF_TOKEN,
        disable_log_stats=True,
    )
except TimeoutError:
    os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_ID,
        max_seq_length=2048,
        load_in_4bit=True,
        token=HF_TOKEN,
        disable_log_stats=True,
    )

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
FastLanguageModel.for_inference(model)
model.eval()

print("[STEP 8.1] Model + LoRA ready.")


[STEP 8] Loading fine-tuned model stack...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

[STEP 8.1] Model + LoRA ready.


In [ ]:
# ============================================================
# STEP 9: Inference Function (copied from usemodelkaggle, non-Gradio)
# ============================================================
print("[STEP 9] Defining evaluator...")

import io
import json

tok = tokenizer
mdl = model
mdl.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
gen_lock = Lock()
history = []

_DRIVE_API = None


def _init_drive_api():
    global _DRIVE_API
    if _DRIVE_API is not None:
        return _DRIVE_API

    try:
        from google.colab import auth
        auth.authenticate_user()
    except Exception:
        # Non-Colab env may still have ADC credentials.
        pass

    try:
        import google.auth
        from googleapiclient.discovery import build
        creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
        _DRIVE_API = build("drive", "v3", credentials=creds, cache_discovery=False)
        return _DRIVE_API
    except Exception as e:
        print(f"[STEP 9 WARN] Drive API init failed: {e}")
        return None


def _extract_drive_id_from_url(url: str) -> str:
    if not url:
        return ""
    patterns = [
        r"/d/e/([a-zA-Z0-9_-]+)",
        r"/d/([a-zA-Z0-9_-]+)",
        r"[?&]id=([a-zA-Z0-9_-]+)",
    ]
    for pat in patterns:
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return ""


def _read_google_shortcut_meta(path: str):
    """
    Read .gdoc/.gsheet/.gslides shortcut metadata.
    Supports JSON shortcut files and INI-style URL shortcuts.
    Returns (file_id, url).
    """
    url = ""
    file_id = ""

    raw = ""
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            raw = f.read()
    except Exception:
        return "", ""

    # Try JSON first
    try:
        meta = json.loads(raw)
        if isinstance(meta, dict):
            url = str(meta.get("url", "") or "")
            file_id = str(meta.get("doc_id", "") or "")

            resource_id = str(meta.get("resource_id", "") or "")
            if not file_id and resource_id:
                # examples: "document/FILE_ID" or "document:FILE_ID"
                if "/" in resource_id:
                    file_id = resource_id.split("/")[-1]
                elif ":" in resource_id:
                    file_id = resource_id.split(":")[-1]

            if not file_id:
                file_id = _extract_drive_id_from_url(url)

            if file_id:
                return file_id, url
    except Exception:
        pass

    # Fallback for non-JSON shortcut content
    # e.g. [InternetShortcut] URL=https://docs.google.com/document/d/...
    m = re.search(r"(?im)^URL\s*=\s*(https?://\S+)", raw)
    if m:
        url = m.group(1).strip()
    else:
        m2 = re.search(r"https?://\S+", raw)
        if m2:
            url = m2.group(0).strip()

    if not file_id and url:
        file_id = _extract_drive_id_from_url(url)

    return file_id, url


def _google_mime_for_shortcut_ext(ext: str) -> str:
    ext = ext.lower()
    mapping = {
        ".gdoc": "application/vnd.google-apps.document",
        ".gsheet": "application/vnd.google-apps.spreadsheet",
        ".gslides": "application/vnd.google-apps.presentation",
        ".gdraw": "application/vnd.google-apps.drawing",
        ".gform": "application/vnd.google-apps.form",
        ".gsite": "application/vnd.google-apps.site",
    }
    return mapping.get(ext, "")


def _lookup_google_file_id_by_name(local_path: str):
    """
    Fallback when .g* shortcut metadata has no file id.
    Lookup by base filename in user's Drive.
    Returns (file_id, webViewLink).
    """
    svc = _init_drive_api()
    if svc is None:
        return "", ""

    stem = os.path.splitext(os.path.basename(local_path))[0]
    ext = os.path.splitext(local_path)[1].lower()
    mime = _google_mime_for_shortcut_ext(ext)

    safe_name = stem.replace("'", "\'")
    q = f"name = '{safe_name}' and trashed = false"
    if mime:
        q += f" and mimeType = '{mime}'"

    try:
        resp = svc.files().list(
            q=q,
            corpora="user",
            pageSize=10,
            fields="files(id,name,mimeType,webViewLink)",
        ).execute()
        files = resp.get("files", [])
        if files:
            return files[0].get("id", ""), files[0].get("webViewLink", "")
    except Exception:
        pass

    # broad fallback: drop mime filter
    try:
        resp = svc.files().list(
            q=f"name = '{safe_name}' and trashed = false",
            corpora="user",
            pageSize=10,
            fields="files(id,name,mimeType,webViewLink)",
        ).execute()
        files = resp.get("files", [])
        if files:
            return files[0].get("id", ""), files[0].get("webViewLink", "")
    except Exception:
        pass

    return "", ""


def _preferred_export_mimes(ext: str):
    ext = ext.lower()
    if ext == ".gsheet":
        return ["text/csv", "application/pdf", "text/plain"]
    if ext == ".gslides":
        return ["text/plain", "application/pdf"]
    if ext == ".gdraw":
        return ["application/pdf", "image/svg+xml", "text/plain"]
    if ext == ".gdoc":
        return ["text/plain", "application/pdf"]
    # generic .g* fallback
    return ["text/plain", "application/pdf", "text/csv"]


def _decode_export_bytes(data, mime_type: str):
    if isinstance(data, str):
        return data

    if mime_type == "application/pdf":
        import PyPDF2
        reader = PyPDF2.PdfReader(io.BytesIO(data))
        return "\n".join([page.extract_text() or "" for page in reader.pages])

    for enc in ("utf-8", "latin-1"):
        try:
            return data.decode(enc, errors="ignore")
        except Exception:
            pass
    return str(data)


def read_google_workspace_file(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    if not ext.startswith('.g'):
        return ""

    file_id, web_url = _read_google_shortcut_meta(path)

    # fallback by Drive lookup if shortcut meta didn't include file id
    if not file_id:
        looked_id, looked_url = _lookup_google_file_id_by_name(path)
        if looked_id:
            file_id = looked_id
            if not web_url:
                web_url = looked_url

    if not file_id:
        return f"(Google shortcut found but file id not found: {path})"

    svc = _init_drive_api()
    if svc is None:
        return f"(Drive API not available for reading {path})"

    last_err = None
    for mime in _preferred_export_mimes(ext):
        try:
            payload = svc.files().export(fileId=file_id, mimeType=mime).execute()
            text = _decode_export_bytes(payload, mime)
            if text and text.strip():
                return text
        except Exception as e:
            last_err = e

    # metadata fallback
    try:
        meta = svc.files().get(fileId=file_id, fields="name,mimeType,webViewLink").execute()
        return (
            f"(Could not export content. name={meta.get('name')} mime={meta.get('mimeType')} "
            f"link={meta.get('webViewLink', web_url)})"
        )
    except Exception:
        return f"(Could not export Google file content. Last error: {last_err})"




def format_output(raw):
    text = raw.replace("####", "").replace("**", "")
    sections = {
        "verdict": r"(?i)verdict[:\-]\s*(.*?)(?=\n|score|benefits|$)",
        "score": r"(?i)score[:\-]\s*([0-9\.]+)",
        "benefits": r"(?i)benefits?[:\-]\s*(.*?)(?=\n\s*risks?|\n\s*overall|$)",
        "risks": r"(?i)risks?[:\-]\s*(.*?)(?=\n\s*overall|\n\s*rationale|$)",
        "overall": r"(?i)(?:overall|rationale|analysis)[:\-]\s*(.*)",
    }
    formatted = {}
    for key, pattern in sections.items():
        m = re.search(pattern, text, re.S)
        formatted[key] = m.group(1).strip() if m else ""
    return formatted


# ===== Long-input summary gate (adapted from Dataset Prep) =====
SCRIPT_SUMMARY_TRIGGER_TOKENS = 2048
SCRIPT_REVIEWER_MAX_PROMPT_TOKENS = 2048
SCRIPT_SUMMARY_TARGETS = ["900-1200", "500-700", "300-450"]
SCRIPT_SUMMARY_MODEL = os.getenv("SCRIPT_SUMMARY_MODEL", "gemini-3-flash-preview")
SCRIPT_SUMMARY_MODEL_FALLBACK = os.getenv("SCRIPT_SUMMARY_MODEL_FALLBACK", "gemini-2.5-flash-lite")


def _count_local_tokens(text: str) -> int:
    text = text or ""
    if not text:
        return 0

    try:
        enc = tok(text, add_special_tokens=False)
        ids = enc["input_ids"] if isinstance(enc, dict) else enc.input_ids
        if ids and isinstance(ids[0], list):
            return len(ids[0])
        return len(ids)
    except Exception:
        try:
            return len(tok.encode(text))
        except Exception:
            return max(1, len(text) // 4)


def _summary_system_instruction(target_token_range: str) -> str:
    # Structure copied from Dataset Prep ScriptSummarizerAgent.
    return f"""
Role: You are a Structural Script Analyst and Narrative Architect.
Task: Compress the provided raw script treatment into a "Uplift-Ready Structural Summary."
Constraints:

Length: Target {target_token_range} tokens.
Preserve: Core conflict, protagonist's internal and external arcs, emotional peaks, and moral/thematic undercurrents.
Remove: Detailed dialogue, minor characters with no plot impact, specific scene descriptions, and fluff.
Specific Focus (The "Mission Keeper" Lens):

Highlight the Protagonist's Agency: How they drive the plot through choices.
Identify Heroine's Journey markers: Internal awakening and reclaiming of power.
Emphasize Moral Dilemmas: Points where the "Humanity Uplifting" principles are tested.
Output Structure (Markdown Headers):

## Logline: A one-sentence summary.
## Thematic Core: The central moral question or "Uplift" theme.
## Character Profile: The protagonist's initial state vs. final state (Arc).
## Structural Breakdown: Key beats (Inciting Incident, Midpoint, Climax, Resolution) with emphasis on emotional transitions.
## Social Context: Any racial, cultural, or gender-specific dynamics relevant to "Dignity & Inclusion."
"""


def _run_script_summary(raw_script: str, filename: str, target_token_range: str):
    prompt = f"Here is the raw script treatment for '{filename}'. Please summarize it:\n\n{raw_script}"

    candidates = []
    for model_name in [
        SCRIPT_SUMMARY_MODEL,
        SCRIPT_SUMMARY_MODEL_FALLBACK,
        "gemini-2.5-flash-lite",
        "gemini-1.5-flash",
    ]:
        if model_name and model_name not in candidates:
            candidates.append(model_name)

    last_err = None
    client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY", ""))
    for model_name in candidates:
        try:
            full_prompt = _summary_system_instruction(target_token_range).strip() + "\n\n---\n\n" + prompt
            response = client.models.generate_content(
                model=model_name,
                contents=full_prompt,
            )
            text = (getattr(response, "text", "") or "").strip()
            if text:
                return text, model_name
        except Exception as e:
            last_err = e
            print(f"[STEP 9 WARN] Summary call failed on {model_name}: {e}")

    print(f"[STEP 9 WARN] Summary generation failed. Last error: {last_err}")
    return "", ""


def _build_reviewer_prompt(story_body: str, ctx: str) -> str:
    template = """Below is an instruction that describes a task.

### Instruction:
Evaluate whether the following idea uplifts humanity. Provide:
- Verdict (Yes/No)
- Score (1 to 5)
- Benefits
- Risks
- Overall rationale
Be objective.

### Input:
{story}

### Additional Context:
{context}

### Response:
"""
    return template.format(
        story=(story_body or "").strip(),
        context=ctx.strip() if ctx else "None",
    )


def _fit_story_to_reviewer_budget(story_text: str, context_text: str, filename: str):
    original_tokens = _count_local_tokens(story_text)
    current_story = story_text
    summary_used = False
    summary_model_used = ""

    for target in SCRIPT_SUMMARY_TARGETS:
        prompt_now = _build_reviewer_prompt(current_story, context_text)
        prompt_tokens_now = _count_local_tokens(prompt_now)
        if prompt_tokens_now <= SCRIPT_REVIEWER_MAX_PROMPT_TOKENS:
            return {
                "story": current_story,
                "prompt": prompt_now,
                "prompt_tokens": prompt_tokens_now,
                "original_tokens": original_tokens,
                "summary_tokens": _count_local_tokens(current_story),
                "summary_used": summary_used,
                "summary_model": summary_model_used,
                "fits": True,
            }

        # First time over threshold: require summary.
        summarized_text, model_used = _run_script_summary(current_story, filename, target)
        if not summarized_text.strip():
            break

        current_story = summarized_text.strip()
        summary_used = True
        summary_model_used = model_used or summary_model_used

    # Final check after summary attempts.
    final_prompt = _build_reviewer_prompt(current_story, context_text)
    final_prompt_tokens = _count_local_tokens(final_prompt)
    return {
        "story": current_story,
        "prompt": final_prompt,
        "prompt_tokens": final_prompt_tokens,
        "original_tokens": original_tokens,
        "summary_tokens": _count_local_tokens(current_story),
        "summary_used": summary_used,
        "summary_model": summary_model_used,
        "fits": final_prompt_tokens <= SCRIPT_REVIEWER_MAX_PROMPT_TOKENS,
    }


def evaluate_story(story, context, uploaded_file):
    with gen_lock:
        if (not story or story.strip() == "") and not uploaded_file:
            return "Please enter a story or upload a file.", 1, history

        file_text = ""
        if uploaded_file is not None:
            path = uploaded_file.name
            ext = os.path.splitext(path)[1].lower()
            try:
                if ext == ".txt":
                    with open(path, "r", encoding="utf-8", errors="ignore") as f:
                        file_text = f.read()
                elif ext in (".docx", ".doc"):
                    import docx
                    doc = docx.Document(path)
                    file_text = "\n".join([p.text for p in doc.paragraphs])
                elif ext == ".pdf":
                    import PyPDF2
                    with open(path, "rb") as f:
                        reader = PyPDF2.PdfReader(f)
                        file_text = "\n".join([page.extract_text() or "" for page in reader.pages])
                elif ext.startswith('.g'):
                    file_text = read_google_workspace_file(path)
                else:
                    file_text = "(File type not supported)"
            except Exception as e:
                file_text = f"(Error reading file: {str(e)})"

        original_story_text = ((story or "") + "\n\n" + file_text).strip()
        if original_story_text == "":
            return "Input is empty.", 1, history

        file_label = os.path.basename(uploaded_file.name) if uploaded_file is not None else "submission"

        # If long input exceeds threshold, summary will replace original script content.
        fit_result = _fit_story_to_reviewer_budget(
            original_story_text,
            context or "",
            file_label,
        )

        final_story = fit_result["story"]
        prompt = fit_result["prompt"]
        prompt_tokens = fit_result["prompt_tokens"]
        original_tokens = fit_result["original_tokens"]
        summary_tokens = fit_result["summary_tokens"]
        summary_used = fit_result["summary_used"]
        summary_model_used = fit_result["summary_model"]

        print("=" * 100)
        print("[DEBUG] SCRIPT INPUT STATS")
        print(f"original_tokens={original_tokens} | summary_used={summary_used} | summary_tokens={summary_tokens}")
        print(f"prompt_tokens={prompt_tokens} | reviewer_limit={SCRIPT_REVIEWER_MAX_PROMPT_TOKENS}")
        print(f"summary_model={summary_model_used if summary_model_used else 'N/A'}")
        print("=" * 100)

        if not fit_result["fits"]:
            msg = (
                "Input is still too long for script reviewer after summary attempts. "
                f"final_prompt_tokens={prompt_tokens}, limit={SCRIPT_REVIEWER_MAX_PROMPT_TOKENS}. "
                "Please shorten the source or adjust summary settings."
            )
            return msg, 1, history

        print("=" * 100)
        print("[DEBUG] FINAL STORY PREVIEW (first 1500 chars)")
        print(final_story[:1500])
        print("=" * 100)

        inputs = tok(prompt, return_tensors="pt").to(device)
        input_len = int(inputs["input_ids"].shape[1])
        if input_len > SCRIPT_REVIEWER_MAX_PROMPT_TOKENS:
            return (
                f"Prompt token overflow before generation: {input_len} > {SCRIPT_REVIEWER_MAX_PROMPT_TOKENS}.",
                1,
                history,
            )

        outputs = mdl.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.4,
            top_p=0.95,
            do_sample=True,
        )

        prompt_len = inputs["input_ids"].shape[1]
        raw_text = tok.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()
        parsed = format_output(raw_text)

        try:
            val = float(parsed["score"])
            final_score = int(round(val))
            final_score = max(1, min(5, final_score))
        except Exception:
            final_score = 1

        history.append({
            "story": (final_story[:200] + "...") if len(final_story) > 200 else final_story,
            "score": final_score,
            "verdict": parsed["verdict"],
            "output": (raw_text[:300] + "...") if len(raw_text) > 300 else raw_text,
            "summary_used": summary_used,
            "summary_model": summary_model_used,
            "prompt_tokens": prompt_tokens,
        })

        display_text = f"""
### Verdict
{parsed['verdict']}

### Score
{parsed['score'] if parsed['score'] else final_score}

### Benefits
{parsed['benefits']}

### Risks
{parsed['risks']}

### Overall Rationale
{parsed['overall']}
""".strip()

        if summary_used:
            display_text = (
                f"_Long input summarized before review ({original_tokens} -> {summary_tokens} tokens; "
                f"model: {summary_model_used})_\n\n" + display_text
            )

        return display_text, final_score, history

print("[STEP 9.1] Evaluator ready.")


[STEP 9] Defining evaluator...
[STEP 9.1] Evaluator ready.


## STEP 10 (Skip in UI Flow)

This one-shot test step is deprecated for the Gradio pipeline.

Use this flow instead:
1. Run `STEP 1` to `STEP 9` once.
2. Run `STEP 11` to build UI.
3. Run `STEP 12` to launch UI.
4. Enter prompt + upload file in Gradio, then click **Analyze**.


In [ ]:
# ============================================================
# STEP 11: Gradio Pipeline (route at submit time + upload local file)
# ============================================================
print("[STEP 11] Building Gradio pipeline...")


def _normalize_uploaded_file(uploaded_file):
    """Support different gradio file return shapes."""
    if uploaded_file is None:
        return None
    if isinstance(uploaded_file, str):
        return SimpleNamespace(name=uploaded_file)
    if hasattr(uploaded_file, "name"):
        return SimpleNamespace(name=uploaded_file.name)
    if isinstance(uploaded_file, dict) and uploaded_file.get("name"):
        return SimpleNamespace(name=uploaded_file["name"])
    return None


def route_then_analyze(user_prompt: str, uploaded_file, context: str):
    if not user_prompt or not user_prompt.strip():
        return "Please enter a prompt.", 1, {"error": "empty_prompt"}

    # Routing happens here at submit time.
    routed_agent, route_mode = safe_route(user_prompt)

    file_obj = _normalize_uploaded_file(uploaded_file)

    # If route says impact, stop here and do NOT run model inference.
    if routed_agent == orchestrator.IMPACT_AGENT:
        msg = f"""### Routed Agent
{routed_agent}

Route mode: {route_mode}

Impact agent's job. Session stops.
""".strip()
        return msg, 1, {
            "routed_agent": routed_agent,
            "route_mode": route_mode,
            "uploaded_file": getattr(file_obj, "name", None),
            "stopped": True,
            "reason": "impact_agent_route",
        }

    # Only script reviewer path continues to usemodel inference.
    out_text, out_score, _ = evaluate_story(
        user_prompt,
        context or "",
        file_obj,
    )

    final_md = f"""### Routed Agent
{routed_agent}

Route mode: {route_mode}

{out_text}
""".strip()

    debug = {
        "routed_agent": routed_agent,
        "route_mode": route_mode,
        "uploaded_file": getattr(file_obj, "name", None),
        "score": out_score,
        "stopped": False,
    }
    return final_md, out_score, debug


default_prompt = globals().get(
    "USER_PROMPT",
    "Please review my script opening and tell me what to improve.",
)
default_context = globals().get(
    "ADDITIONAL_CONTEXT",
    "Focus on clarity, pacing, and emotional impact.",
)

with gr.Blocks() as demo:
    gr.Markdown("# Orchestrator + UseModel (Gradio)")
    gr.Markdown("Upload a local file (.pdf/.doc/.docx/.txt), then click Analyze.")

    user_prompt_box = gr.Textbox(
        label="User Prompt",
        lines=4,
        value=default_prompt,
    )

    context_box = gr.Textbox(
        label="Additional Context (optional)",
        lines=3,
        value=default_context,
    )

    file_box = gr.File(
        label="Upload Script File",
        file_types=[".pdf", ".doc", ".docx", ".txt"],
    )

    analyze_btn = gr.Button("Analyze", variant="primary")

    result_md = gr.Markdown(label="Result")
    score_slider = gr.Slider(label="Score", minimum=1, maximum=5, step=1, value=1, interactive=False)
    debug_json = gr.JSON(label="Debug")

    analyze_btn.click(
        fn=route_then_analyze,
        inputs=[user_prompt_box, file_box, context_box],
        outputs=[result_md, score_slider, debug_json],
    )

print("[STEP 11.1] Gradio pipeline ready. Run STEP 12 to launch.")


[STEP 11] Building Gradio pipeline...
[STEP 11.1] Gradio pipeline ready. Run STEP 12 to launch.


In [ ]:
# ============================================================
# STEP 12: Launch Gradio
# ============================================================
print("[STEP 12] Launching Gradio...")
demo.launch(share=True, debug=True, prevent_thread_lock=True)


[STEP 12] Launching Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://18be8f45e0a2d13f23.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



[DEBUG] FINAL STORY (user prompt + uploaded script text)
Please review my script opening and tell me what to improve.

A coalition of teachers, engineers, and local volunteers launches a free community learning center that offers open-access STEM workshops, AI literacy classes, and mentorship for students from underserved backgrounds.
The program runs year-round, provides laptops and materials at no cost, and pairs each student with a trained mentor who supports both academic growth and emotional well-being. Alumni return to become new mentors, creating a self-sustaining cycle of empowerment.
The center is designed to be inclusive for learners of all ages and abilities, and all curriculum materials are open-source so that other communities can replicate the model.
Evaluate whether this initiative uplifts humanity, provide a score from 0 to 1, explain benefits and risks, and give a concise recommendation for scaling.



[DEBUG] FULL PROMPT SENT TO MODEL
Below is an instruction that des

## Default Prompt for Script Reviewer

`Please review my script opening and tell me what to improve.`

## Test Impact Agent's Prompt:

`Please analyze the societal impact and ethical implications of this submission. Focus on community outcomes, potential harms, benefits, and long-term consequences. If this policy is scaled, what would happen to vulnerable groups?`

In [ ]:
demo.close()


NameError: name 'demo' is not defined